In [0]:
--Querying via Unity Catalog
SELECT * FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6");

--Query Data Lineage (See which tables depend on which — the "map" of your pipeline.)
SELECT
  details:flow_definition.output_dataset AS output_dataset,
  details:flow_definition.input_datasets AS input_dataset
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'flow_definition';

--Query Data Quality (Expectations-See pass/fail/warn counts for each expectation)
SELECT
  timestamp,
  details:flow_progress.data_quality.expectations AS expectations
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality IS NOT NULL;

--Monitor Data Backlog (See how much data is waiting to be processed)
SELECT
  timestamp,
  DOUBLE(details:flow_progress.metrics.backlog_bytes) AS backlog_bytes
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'flow_progress';

-- Monitor Enhanced Autoscaling (see when the cluster resize itself)
SELECT * FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'autoscale';

--Monitor Compute Utilization (Check cluster health — how busy are the task slots?)
SELECT
  timestamp,
  DOUBLE(details:cluster_resources.avg_num_queued_tasks) AS queue_size
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'cluster_resources';

--Audit User Actions (who did what and when)
SELECT
  timestamp,
  details:user_action:action AS action,
  details:user_action:user_name AS user_name
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'user_action';

--Get Runtime Version(Which Databricks Runtime version was used?)
SELECT details:create_update:runtime_version:dbr_version
FROM event_log("f44ec8ed-08e9-463b-b9d7-cfd6d8d5f0f6")
WHERE event_type = 'create_update';